In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = (
    SparkSession.builder
    .appName("2_Ingest_MovieLens_To_HDFS")
    .master("spark://spark-master:7077")
    .config("spark.hadoop.fs.defaultFS", "hdfs://namenode:8020")
    .config("spark.driver.host", "notebook")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.executor.memory", "1g")
    .config("spark.driver.memory", "1g")
    .config("spark.executor.cores", "1")
    .config("spark.cores.max", "1")
    .config("spark.executor.instances", "1")
    .config("spark.sql.shuffle.partitions", "16")
    .getOrCreate()
)

LOCAL_RAW_DIR = "/workspace/data/raw/ml-25m"
LOCAL_RAW = f"file://{LOCAL_RAW_DIR}"
HDFS_RAW = "hdfs://namenode:8020/netflix-recsys/raw/ml-25m"

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/05 06:04:44 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/05 06:04:45 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [2]:
ratings = spark.read.csv(f"{LOCAL_RAW}/ratings.csv", header=True, inferSchema=True)
movies = spark.read.csv(f"{LOCAL_RAW}/movies.csv", header=True, inferSchema=True)
tags = spark.read.csv(f"{LOCAL_RAW}/tags.csv", header=True, inferSchema=True)
links = spark.read.csv(f"{LOCAL_RAW}/links.csv", header=True, inferSchema=True)

In [3]:
print("ratings:", ratings.count())
print("movies:", movies.count())
print("tags:", tags.count())
print("links:", links.count())

ratings.printSchema()
movies.printSchema()

ratings: 25000095
movies: 62423
tags: 1093360
links: 62423
root
 |-- userId: integer (nullable = true)
 |-- movieId: integer (nullable = true)
 |-- rating: double (nullable = true)
 |-- timestamp: integer (nullable = true)

root
 |-- movieId: integer (nullable = true)
 |-- title: string (nullable = true)
 |-- genres: string (nullable = true)



In [4]:
ratings.write.mode("overwrite").parquet(f"{HDFS_RAW}/ratings")
movies.write.mode("overwrite").parquet(f"{HDFS_RAW}/movies")
tags.write.mode("overwrite").parquet(f"{HDFS_RAW}/tags")
links.write.mode("overwrite").parquet(f"{HDFS_RAW}/links")

In [5]:
for path in [
    "hdfs://namenode:8020/netflix-recsys/bronze",
    "hdfs://namenode:8020/netflix-recsys/silver",
    "hdfs://namenode:8020/netflix-recsys/gold",
    "hdfs://namenode:8020/netflix-recsys/models",
    "hdfs://namenode:8020/netflix-recsys/outputs",
]:
    spark._jvm.org.apache.hadoop.fs.FileSystem.get(
        spark._jsc.hadoopConfiguration()
    ).mkdirs(
        spark._jvm.org.apache.hadoop.fs.Path(path)
    )